In [2]:
from google.colab import userdata
import os
import requests
import time

# --- 1. Clone the repository into 'leetcode_programming3' ---
!git clone https://github.com/aqwertyuiop48/leetcode_programming.git leetcode_programming3 || true

# Setup your final destination inside the new folder
final_dir = "/content/leetcode_programming3/java_solutions"
os.makedirs(final_dir, exist_ok=True)

# --- 2. Get cookie and extract CSRF Token ---
try:
    user_cookie = userdata.get('LEETCODE_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("❌ Secret 'LEETCODE_TOKEN' not found!")

csrf_token = ""
for item in user_cookie.split(';'):
    if 'csrftoken' in item:
        csrf_token = item.split('=')[1].strip()

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0',
    'Cookie': user_cookie,
    'x-csrftoken': csrf_token,
    'Referer': 'https://leetcode.com/'
})

# --- 3. Fetch the Problem List to map Slugs to IDs ---
print("\n🔍 Fetching LeetCode problem dictionary...")
resp = session.get("https://leetcode.com/api/problems/all/")

if resp.status_code != 200:
    print(f"❌ Authentication failed (Status {resp.status_code}). Your token might be expired.")
else:
    problem_info = {}
    for p in resp.json().get('stat_status_pairs', []):
        slug = p['stat']['question__title_slug']
        problem_info[slug] = {
            'id': p['stat']['frontend_question_id'],
            'title': p['stat']['question__title']
        }

    # --- 4. Fetch Submissions & Overwrite Originals ---
    print("🚀 Downloading Java submissions and updating original files...\n")
    offset = 0
    limit = 20
    seen_slugs = set()
    count = 0

    while True:
        url = f"https://leetcode.com/api/submissions/?offset={offset}&limit={limit}&status_display=Accepted"

        # Rate limit protection
        retries = 0
        while retries < 4:
            resp = session.get(url)
            if resp.status_code == 200:
                break
            print(f"⚠️ Rate limit hit. Cooling down for 10s...")
            time.sleep(10)
            retries += 1

        if resp.status_code != 200:
            print("❌ API Error. Stopping to prevent IP ban.")
            break

        submissions = resp.json().get('submissions_dump', [])
        if not submissions:
            break # Reached the end of your history

        for sub in submissions:
            # Target Java only, newest submission per problem
            if sub['lang'] == 'java' and sub['title_slug'] not in seen_slugs:
                slug = sub['title_slug']
                seen_slugs.add(slug)

                info = problem_info.get(slug, {'id': 0, 'title': slug})
                prob_id = info['id']
                title = info['title']

                # NO MORE "COPY" SUFFIX - We target the original file directly
                filename = f"{prob_id}.{slug}.java"
                filepath = os.path.join(final_dir, filename)

                # Default metadata format
                metadata = f"/*\n * @lc app=leetcode id={prob_id} lang=java\n *\n * [{prob_id}] {title}\n */\n\n"

                # Check if the original blank file exists to preserve its metadata
                if os.path.exists(filepath):
                    with open(filepath, "r", encoding="utf-8") as f:
                        existing_content = f.read()
                        end_of_comment = existing_content.find("*/")

                        if end_of_comment != -1:
                            metadata = existing_content[:end_of_comment + 2] + "\n\n"
                            print(f"🔄 Overwrote: {filename} (Kept existing metadata)")
                        else:
                            print(f"🔄 Overwrote: {filename} (Generated new metadata)")
                else:
                    print(f"✅ Created: {filename}")

                # Write metadata + the raw code back into the original file
                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(metadata + sub['code'].strip() + "\n")

                count += 1

        offset += limit
        time.sleep(2.5) # Crucial sleep to avoid LeetCode bans

    print(f"\n🎉 Finished downloading {count} files!")

    # --- 5. Clean up old "copy" files just in case they were left behind ---
    print("\n🧹 Sweeping folder for old 'copy' files...")
    deleted = 0
    for file in os.listdir(final_dir):
        if file.endswith(" copy.java"):
            os.remove(os.path.join(final_dir, file))
            deleted += 1

    if deleted > 0:
        print(f"🗑️ Deleted {deleted} outdated copy files to keep your repo clean.")
    else:
        print("✨ Folder is perfectly clean. No duplicates found.")

    print(f"📁 All final solutions are neatly stored in: {final_dir}")

fatal: destination path 'leetcode_programming3' already exists and is not an empty directory.

🔍 Fetching LeetCode problem dictionary...
🚀 Downloading Java submissions and updating original files...

🔄 Overwrote: 4.median-of-two-sorted-arrays.java (Kept existing metadata)
🔄 Overwrote: 29.divide-two-integers.java (Kept existing metadata)
🔄 Overwrote: 2296.design-a-text-editor.java (Kept existing metadata)
🔄 Overwrote: 1912.design-movie-rental-system.java (Kept existing metadata)
🔄 Overwrote: 2102.sequentially-ordinal-rank-tracker.java (Kept existing metadata)
🔄 Overwrote: 2276.count-integers-in-intervals.java (Kept existing metadata)
🔄 Overwrote: 1825.finding-mk-average.java (Kept existing metadata)
🔄 Overwrote: 2642.design-graph-with-shortest-path-calculator.java (Kept existing metadata)
🔄 Overwrote: 30.substring-with-concatenation-of-all-words.java (Kept existing metadata)
🔄 Overwrote: 68.text-justification.java (Kept existing metadata)
🔄 Overwrote: 124.binary-tree-maximum-path-sum.ja

In [3]:
import shutil
from google.colab import files

print("📦 Zipping your repository...")
# This creates a file called 'my_leetcode_repo.zip' containing your folder
shutil.make_archive("/content/my_leetcode_repo", 'zip', "/content/leetcode_programming3")

print("⬇️ Triggering download to your computer...")
# This forces your browser to download the zip file
files.download("/content/my_leetcode_repo.zip")

📦 Zipping your repository...
⬇️ Triggering download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>